In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name,col
from delta.tables import DeltaTable


In [0]:
# Vérifier l'état actuel de Bronze

df_before = spark.table(
    "training_catalog.bronze.ecommerce_bronze"
)

print("Nombre total de lignes :", df_before.count())

print(
    "Nombre de commandes uniques :",
    df_before.select("order_id").distinct().count()
)

In [0]:
# READ JOB PARAMETERS
# Source folder for incremental ingestion
dbutils.widgets.text(
    "source_path",
    "/Volumes/training_catalog/christelle_schema/raw_files/"
)

# Catalog
dbutils.widgets.text(
    "catalog_name",
    "training_catalog"
)

# Get parameters
source_path = dbutils.widgets.get("source_path")
catalog_name = dbutils.widgets.get("catalog_name")

print("Source folder :", source_path)
print("Catalog       :", catalog_name)

In [0]:
# AUTO LOADER - INCREMENTAL INGESTION

schema_path = (
    "/Volumes/training_catalog/christelle_schema/"
    "raw_files/_schemas/ecommerce"
)

df_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
)

In [0]:
# ANCIENNE LECTURE - FULL LOAD
# df_raw = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv(source_file)
# )

In [0]:
# Schemas validation
df_raw.printSchema()

In [0]:
# ajout des matadata d'ingestion
df_bronze = (
    df_raw
    .withColumn(
        "ingest_file_name",
        col("_metadata.file_path")
    )
    .withColumn(
        "ingest_time",
        current_timestamp()
    )
)

In [0]:
#write the raw validated data to the bronze
#bronze_table = f"{catalog_name}.bronze.ecommerce_bronze"
#df_bronze.write.format("delta").mode("overwrite").saveAsTable( "bronze_table")

In [0]:
target_table = "training_catalog.bronze.ecommerce_bronze"

def merge_new_orders(microbatch_df, batch_id):

    # Garder un seul enregistrement par order_id
    new_orders = microbatch_df.dropDuplicates(["order_id"])

    # Accéder à la table Bronze existante
    target = DeltaTable.forName(
        spark,
        target_table
    )

    # Insérer uniquement les order_id qui n'existent pas
    (
        target.alias("target")
        .merge(
            new_orders.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
checkpoint_path = (
    "/Volumes/training_catalog/christelle_schema/"
    "raw_files/_checkpoints/ecommerce_bronze"
)

query = (
    df_bronze.writeStream
    .foreachBatch(merge_new_orders)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
query = (
    df_bronze.writeStream
    .foreachBatch(merge_new_orders)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

